## Datenbereinigung für Datenbank Airbnb NYC

PGAdmin/PostgreSQL


SQL für Table in PGAdmin: 

CREATE TABLE nyc_airbnb_listings_raw (
    -- 1. IDENTIFIKATION UND LAGE
    id BIGINT PRIMARY KEY,
    city VARCHAR(50) NOT NULL, -- Für die spätere Mehrfachnutzung
    latitude DOUBLE PRECISION NOT NULL,
    longitude DOUBLE PRECISION NOT NULL,

    -- 2. ZIELVARIABLE UND KATEGORIEN (als VARCHAR gespeichert!)
    price NUMERIC(10, 2) NOT NULL,
    room_type VARCHAR(50) NOT NULL, 
    neighbourhood_group_cleansed VARCHAR(50), 
    
    -- 3. UNTERKUNFT-SPEZIFISCHE FEATURES
    accommodates INTEGER NOT NULL,
    bedrooms INTEGER,
    beds INTEGER,
    bathrooms NUMERIC(3, 1), -- Typischerweise 1.0, 1.5, etc.
    minimum_nights INTEGER NOT NULL,
    availability_365 INTEGER NOT NULL,

    -- 4. BINÄRE UND HOST-INFOS
    instant_bookable BOOLEAN NOT NULL,
    host_is_superhost BOOLEAN, -- Lässt NULL zu, falls Info fehlt
    host_total_listings_count INTEGER,

    -- 5. BEWERTUNGEN
    review_scores_rating NUMERIC(4, 2) -- Lässt NULL zu
);

In [18]:
import pandas as pd



In [19]:
import sys
import os

# Definiert den relativen Pfad zum 'src' Ordner
# .. = Ein Ordner hoch zum Projekt-Root
# /src = Dann in den Ordner src
src_path = os.path.join(os.getcwd(), '..', 'src')

# Fügt diesen Pfad dem Python-Suchpfad hinzu
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"✅ src-Ordner temporär zum Suchpfad hinzugefügt: {src_path}")

✅ src-Ordner temporär zum Suchpfad hinzugefügt: c:\Users\Annette\Data_Science\02_data-analytics\05_regression_projekt_airbnb\notebooks\..\src


In [20]:
df = pd.read_csv("../data/raw/listings_NYC.csv")

In [21]:
df.shape

(36111, 79)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36111 entries, 0 to 36110
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            36111 non-null  int64  
 1   listing_url                                   36111 non-null  object 
 2   scrape_id                                     36111 non-null  int64  
 3   last_scraped                                  36111 non-null  object 
 4   source                                        36111 non-null  object 
 5   name                                          36109 non-null  object 
 6   description                                   35153 non-null  object 
 7   neighborhood_overview                         18704 non-null  object 
 8   picture_url                                   36111 non-null  object 
 9   host_id                                       36111 non-null 

In [23]:
print(df.columns.tolist())
#print("-" * 50)

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availabil

## Schritt 1: Unnötige Spalten entfernen

In [24]:
SQL_COLUMNS = [
    'id',
    'price',
    'accommodates',
    'room_type',
    'bedrooms',
    'beds',
    'bathrooms',  # Muss vorhanden sein (ggf. aus 'bathrooms_text' berechnet)
    'neighbourhood_group_cleansed',
    'minimum_nights',
    'instant_bookable',
    'review_scores_rating',
    'host_is_superhost',
    'host_total_listings_count',
    'availability_365',
    'latitude',
    'longitude'

]

df1 = df[SQL_COLUMNS].copy()

In [25]:
df1.head()


,id,price,accommodates,room_type,bedrooms,beds,bathrooms,neighbourhood_group_cleansed,minimum_nights,instant_bookable,review_scores_rating,host_is_superhost,host_total_listings_count,availability_365,latitude,longitude
0,40824219,$66.00,1,Private room,1.0,1.0,1.0,Queens,30,f,4.81,t,3.0,77,40.74698,-73.91763
1,40833186,NaN,2,Private room,1.0,NaN,NaN,Manhattan,30,t,NaN,f,1.0,0,40.72314,-73.99323
2,40837137,NaN,1,Private room,NaN,NaN,NaN,Brooklyn,30,f,5.00,f,1.0,0,40.64607,-74.00552
3,40838018,NaN,1,Entire home/apt,1.0,NaN,NaN,Brooklyn,90,t,5.00,f,3.0,0,40.68370,-73.96115
4,40839416,$76.00,1,Private room,1.0,1.0,2.0,Manhattan,30,f,4.95,t,16.0,168,40.72147,-73.98270


In [26]:
df1.tail()

,id,price,accommodates,room_type,bedrooms,beds,bathrooms,neighbourhood_group_cleansed,minimum_nights,instant_bookable,review_scores_rating,host_is_superhost,host_total_listings_count,availability_365,latitude,longitude
36106,1518145740023617475,$175.00,4,Entire home/apt,2.0,1.0,1.0,Manhattan,30,t,NaN,f,49.0,153,40.721930,-73.986668
36107,1518194647322488321,$249.00,4,Entire home/apt,1.0,1.0,1.0,Manhattan,30,t,NaN,t,17.0,326,40.777123,-73.956899
36108,1518232643177537314,$241.00,4,Entire home/apt,1.0,1.0,1.0,Manhattan,30,f,NaN,t,20.0,336,40.777913,-73.957333
36109,1518345023898568625,$170.00,3,Entire home/apt,1.0,1.0,1.0,Manhattan,30,f,NaN,f,261.0,335,40.784140,-73.974890
36110,1518352509290380578,$137.00,2,Entire home/apt,0.0,1.0,1.0,Manhattan,30,f,NaN,f,261.0,341,40.782350,-73.974190


## Schritt 2: Zeilen ohne Preis entfernen und Datentyp anpassen

In [27]:
df2 = df1.dropna(subset=['price']).copy()
print(f"Zeilen nach der Preislöschung: {len(df2)}")

Zeilen nach der Preislöschung: 21328


In [28]:
# 1. Entfernen von Währungszeichen ($), Kommas (,) und Leerzeichen
df2['price'] = df2['price'].astype(str).str.replace(r'[$,\s]', '', regex=True)

# 2. Konvertieren zu numerischem Float
# Wichtig: errors='coerce' verwandelt Einträge, die nicht konvertiert werden können (z.B. leere Zellen, 'N/A'), in NaN.
df2['price'] = pd.to_numeric(df2['price'], errors='coerce')

# 3b. NEU: Nachbereinigung, falls 'coerce' neue NaN-Werte erzeugt hat
rows_before = len(df2)
df2.dropna(subset=['price'], inplace=True)
rows_after = len(df2)

if rows_before > rows_after:
    print(f"⚠️ ACHTUNG: {rows_before - rows_after} Zeilen wurden nach der String-Bereinigung entfernt, da sie nicht numerisch waren.")
    print(f"Neue Zeilenanzahl: {rows_after}")

## Schritt 3: Binare Spalten konvertieren

Spalten: instant_bookable, host_is_superhost

In [33]:
from IPython.display import display
display(df2[['instant_bookable', 'host_is_superhost']].head(20))

,instant_bookable,host_is_superhost
0,False,t
4,False,t
5,False,t
7,False,t
8,False,f
9,False,f
10,False,t
11,False,f
12,False,f
14,True,t


In [31]:
unique_values = df2['instant_bookable'].unique()
print("Eindeutige Werte in 'instant_bookable':", unique_values)
print("--- Vollständige Verteilung in 'instant_bookable' (t, f, NaN) ---")
print(df2['instant_bookable'].value_counts(dropna=False))

Eindeutige Werte in 'instant_bookable': ['f' 't']
--- Vollständige Verteilung in 'instant_bookable' (t, f, NaN) ---
instant_bookable
f    15966
t     5362
Name: count, dtype: int64


In [35]:
unique_values1 = df2['host_is_superhost'].unique()
print("Eindeutige Werte in 'host_is_superhost':", unique_values1)
print("--- Vollständige Verteilung in 'host_is_superhost' (t, f, NaN) ---")
print(df2['host_is_superhost'].value_counts(dropna=False))

Eindeutige Werte in 'host_is_superhost': ['t' 'f' nan]
--- Vollständige Verteilung in 'host_is_superhost' (t, f, NaN) ---
host_is_superhost
f      14624
t       6395
NaN      309
Name: count, dtype: int64


a) Umwandlung von 'instant_bookable' in booleschen Typ

In [32]:
 # MAPPING: 't' und 'f' auf True und False mappen
boolean_mapping = {'t': True, 'f': False}

# KONVERTIERUNG: Die .map-Funktion wendet das Dictionary an
# und behält gleichzeitig alle NaN-Werte bei, was gut ist für Spalten, 
# die NULL zulassen (wie host_is_superhost).
df2['instant_bookable'] = df2['instant_bookable'].map(boolean_mapping)

# PRÜFUNG: Kontrolle, welche Werte jetzt vorhanden sind
print("Eindeutige Werte in 'instant_bookable':")
print(df2['instant_bookable'].value_counts(dropna=False))

Eindeutige Werte in 'instant_bookable':
instant_bookable
False    15966
True      5362
Name: count, dtype: int64


b) Umwandlung von 'host_is_superhost' in Boolean

In [36]:
# MAPPING: 't' und 'f' auf True und False mappen
boolean_mapping1 = {'t': True, 'f': False}

# KONVERTIERUNG: Die .map-Funktion wendet das Dictionary an
# und behält gleichzeitig alle NaN-Werte bei, was gut ist für Spalten, 
# die NULL zulassen.
df2['host_is_superhost'] = df2['host_is_superhost'].map(boolean_mapping1)

# PRÜFUNG: Kontrolle, welche Werte jetzt vorhanden sind
print("Eindeutige Werte in 'host_is_superhost':")
print(df2['host_is_superhost'].value_counts(dropna=False))

Eindeutige Werte in 'host_is_superhost':
host_is_superhost
False    14624
True      6395
NaN        309
Name: count, dtype: int64


## Schritt 4: Spalte 'city' ergänzen

In [39]:
df2['city'] = 'NYC'
print("--- Die ersten Zeilen mit der neuen 'city'-Spalte ---")
print(df2[['city', 'price', 'instant_bookable', 'host_is_superhost']].head())

--- Die ersten Zeilen mit der neuen 'city'-Spalte ---
  city  price  instant_bookable host_is_superhost
0  NYC   66.0             False              True
4  NYC   76.0             False              True
5  NYC   97.0             False              True
7  NYC   60.0             False              True
8  NYC  425.0             False             False


## Schritt 5: Bathrooms bereinigen

In [40]:
df2['bathrooms'].unique()

array([ 1. ,  2. ,  4. ,  1.5,  3. ,  2.5,  5. ,  0. ,  nan,  3.5,  0.5,
       15.5, 10.5,  4.5,  5.5,  6. ,  7. ,  7.5,  9. ,  6.5])

In [43]:
df2['bathrooms'].isnull().sum()

np.int64(7)

In [44]:
df2['bathrooms'] = pd.to_numeric(df2['bathrooms'], errors='coerce')
print(f"Datentyp von 'bathrooms' nach Konvertierung: {df2['bathrooms'].dtype}")

Datentyp von 'bathrooms' nach Konvertierung: float64


## Schritt 6: Finale Überprüfung vor Einpflege in die DB

In [46]:
df2.shape

(21328, 17)

In [49]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21328 entries, 0 to 36110
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            21328 non-null  int64  
 1   price                         21328 non-null  float64
 2   accommodates                  21328 non-null  int64  
 3   room_type                     21328 non-null  object 
 4   bedrooms                      21242 non-null  float64
 5   beds                          21287 non-null  float64
 6   bathrooms                     21321 non-null  float64
 7   neighbourhood_group_cleansed  21328 non-null  object 
 8   minimum_nights                21328 non-null  int64  
 9   instant_bookable              21328 non-null  bool   
 10  review_scores_rating          14944 non-null  float64
 11  host_is_superhost             21019 non-null  object 
 12  host_total_listings_count     20284 non-null  float64
 13  availa

## Export in Ordner Data

In [50]:
import os

# 1. Definieren des Zielpfads
base_path = r'C:\Users\Annette\Data_Science\02_data-analytics\05_regression_projekt_airbnb\data'
file_name = 'nyc_airbnb_prepared_for_sql.csv'
target_folder = 'interim'

# Den vollständigen Pfad erstellen (nutzt os.path.join für korrekte Schrägstriche)
full_path = os.path.join(base_path, target_folder, file_name)

# 2. Export des DataFrames
# index=False ist WICHTIG, damit Pandas die Zeilennummern (den Index) 
# NICHT als zusätzliche Spalte in die CSV-Datei schreibt.
df2.to_csv(full_path, index=False)

print("--- Export-Bestätigung ---")
print(f"✅ DataFrame wurde erfolgreich exportiert nach:")
print(full_path)

--- Export-Bestätigung ---
✅ DataFrame wurde erfolgreich exportiert nach:
C:\Users\Annette\Data_Science\02_data-analytics\05_regression_projekt_airbnb\data\interim\nyc_airbnb_prepared_for_sql.csv


In [55]:
df_check = pd.read_csv("../data/interim/nyc_airbnb_prepared_for_sql.csv")

In [56]:
df_check.head()

,id,price,accommodates,room_type,bedrooms,beds,bathrooms,neighbourhood_group_cleansed,minimum_nights,instant_bookable,review_scores_rating,host_is_superhost,host_total_listings_count,availability_365,latitude,longitude,city
0,40824219,66.0,1,Private room,1.0,1.0,1.0,Queens,30,False,4.81,True,3.0,77,40.746980,-73.917630,NYC
1,40839416,76.0,1,Private room,1.0,1.0,2.0,Manhattan,30,False,4.95,True,16.0,168,40.721470,-73.982700,NYC
2,40843980,97.0,6,Entire home/apt,2.0,3.0,1.0,Queens,30,False,4.14,True,2.0,364,40.682300,-73.845450,NYC
3,40824301,60.0,1,Private room,2.0,1.0,1.0,Brooklyn,30,False,4.92,True,1.0,187,40.713163,-73.943077,NYC
4,40825740,425.0,6,Entire home/apt,3.0,3.0,4.0,Brooklyn,30,False,5.00,False,9.0,224,40.674120,-73.941230,NYC


In [57]:
df_check.columns.tolist()

['id',
 'price',
 'accommodates',
 'room_type',
 'bedrooms',
 'beds',
 'bathrooms',
 'neighbourhood_group_cleansed',
 'minimum_nights',
 'instant_bookable',
 'review_scores_rating',
 'host_is_superhost',
 'host_total_listings_count',
 'availability_365',
 'latitude',
 'longitude',
 'city']

## Export in DB